# High-performance and parallel computing for AI - Practical 7: Multi-CPU and multi-GPU computing in JAX
# MPI4JAX, equinox, pytrees, and profiling JAX code

## NOTE

This practical contains questions related to both the (mpi4jax + pytrees + equinox) part and the JAX profiling part. Depending on how the lectures go we may not have covered both. If that's the case, only do the practicals related to the material covered in the lectures. I will give you directions during class.

## IMPORTANT

* When using a GPU we need to set some environment variables (see below). If you get some weird error restart the kernel and rerun the code below.
* For these practicals and tutorials we will be using a different `conda environment`. When opening a notebook or a terminal make sure you are using the **hpc4ai Kernel**!!!

Before you start (and before running any other GPU code on the servers) please run the following code, which attempts to limit the maximum GPU memory usage and picks two L40s GPUs at random, excluding the Quadro GPUs. **Please only run the code below once every time you restart the kernel!**

**Note:** Here we are setting the `JAX_NUM_CPU_DEVICES` environment variable to tell JAX we want it to see more than one CPU.

**Note:** As of 03/06/2026 `mpi4jax` version `0.9.0.post1` has an installation bug and fails when communicating between GPUs. While the developers fix this we are setting the environment variable `MPI4JAX_USE_CUDA_MPI` to `0`. It will make things work, but all memory will move through the CPU. In the future when they fix it you can check it is fixed by setting `MPI4JAX_USE_CUDA_MPI` to `1`. If it's not fixed it will crash with a segfault.

In [1]:
import os

# JAX-specific environment variables
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.249" # restrict GPU memory preallocation to a fourth of the total
os.environ["JAX_OPTIMIZATION_LEVEL"]="O1"
os.environ["JAX_NUM_CPU_DEVICES"] = "6" # JAX will see 6 CPUs (if available)
os.environ["MPI4JAX_USE_CUDA_MPI"] = "0" # temporary fix while the mpi4jax developers fix the installation script.

## On goliat we have FIVE GPUs so here we pick two of those at random
## so that we do not overload the system.
## The way we do it is by figuring out the GPU UUIDs and then setting
## The CUDA_VISIBLE_DEVICES environment variable.
## Note: this is useful for other libraries as well (e.g., Jax, PyTorch, TF) in multi-GPU servers.

# To get these UUIDs you need to run nvidia-smi -q on the command line
quadro_UUIDs = ["GPU-4efa947b-abbd-7c6e-84f5-61241d34bb4b",
                "GPU-5eb524b0-2b1b-fe98-e6ed-b8fb5185e993"]

L40s_UUIDs = ["GPU-7bba1f33-03d2-016b-d42e-ced83c3ac243",
              "GPU-179d068a-3bea-91d7-1a8c-7017f55d6298",
              "GPU-ae634859-dd49-de46-9182-195639405eaa"]

import random

a, b = random.sample(range(3), 2) # picks two numbers between 0 and 2 at random

# Picks an L40s and a Quadro GPU at random. The others will be invisible to JAX
# NOTE: this only works if the environment variable is set BEFORE JAX is first imported.
os.environ["CUDA_VISIBLE_DEVICES"] = L40s_UUIDs[a] + "," + L40s_UUIDs[b]

## JAX (or whatever other software) will only see these GPUs.

############################ IMPORTS #############################

import jax
import jax.numpy as jnp
import equinox as eqx
from jax.flatten_util import ravel_pytree
import shutil # used for file management to circumvent some jupyterhub remote server issues - not usually needed

gpus = jax.devices("gpu")
cpus = jax.devices("cpu")

assert len(cpus) == 6 and len(gpus) == 2

<hr style="height:4px; background-color:black; border:none; border-color:black;">

# MPI4JAX and Equinox questions

## Parallel training with mpi4jax

We can use `mpi4jax` to train AI models in parallel. The typical strategies are:

* **Data parallel.** Split the data across ranks.
* **Model parallel.** Split the AI model across ranks (e.g., the NN parameters).
* **2D parallelism.** Do both.

You can of course do much more!

**Note:** This tutorial only considers data-parallel training examples. For the other cases, see the JAX sharding practical.

### Data-parallel training with mpi4jax

In data-parallel training each rank holds a different chunk of data and holds an identical copy of the model parameters.

At each training step:

1. Compute local gradient on local data (i.e., the data available on each rank).
2. Use **`mpi4jax.allreduce`** with `MPI.SUM` on the gradient and divide by the MPI size to compute the average gradient.
3. Use the average gradient to take the training step. Since you used allreduce all ranks will hold an identical copy of the average gradient.

We'll see this in the next two exercises in two flavours:

- **Question 1**: parameters are a single 1D array (linear regression). One `allreduce` on a flat array.
- **Question 2**: parameters are a PyTree (an `equinox` MLP). This works in the same way, but requires pytree operations (e.g., `jax.tree.map` or pytree flattening).

### Question 1 — Distributed GD on linear regression

Consider the linear regression model:

$$y = w \cdot x + \eta$$

where $(x,y)$ with $y$ scalar and $x\in\mathbb{R}^n$ are the data we want to fit, $\eta$ is zero-mean, unit-variance Gaussian noise, and $w\in\mathbb{R}^n$ is a vector of model parameters we want to estimate. We will be using a classical $\ell^2$-data misfit loss:

$$ L(w) = \frac{1}{M}\sum_{i=1}^{M} (y_i - w \cdot x_i)^2$$,

where $M$ is the total number of data points $\{(x_i, y_i)\}_{i=1}^M$.

Modify the following code to implement data-parallel gradient descent with learning rate $\gamma = 0.05$ across $4$ MPI ranks.
Use $m=1024$ data points per rank so that $M=\mathrm{mpiSize}\times m$.
Verify the learned weights converge to the exact parameter vector $w_{\mathrm{true}}$.

**Things to notice in the script:**

- `key_true` is the same on every rank (so all ranks know the ground truth `w_true`); `key_data` is distinct (different data per rank).
- `w = jnp.zeros(n)` — the same initial value on every rank, which keeps parameters in sync.
- You only need a single `mpi4jax.allreduce` call.
- Note that here we are only using CPUs, but the code would work the same if we had $4$ GPUs.

**Hints and sanity checks:**

- The loss should decrease monotonically.
- $\lVert w - w_\mathrm{true}\rVert$` should drop to $\approx 0.01$ or lower (the limiting error is set by the noise level).
- If you do not use `allreduce` or if you use different initial conditions, the ranks will *diverge*: print $w$ from each rank at the end and watch them disagree.

In [2]:
%%writefile temp.py
"""Data-parallel SGD on linear regression.

Each rank holds a disjoint subset of training data. After every step,
gradients are averaged across ranks via mpi4jax.allreduce, so every rank
maintains identical weights.
"""

import jax
import jax.numpy as jnp
from mpi4py import MPI
import mpi4jax

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

# Selecting a single CPU and using that by setting the jax_default_device
gpus = jax.devices("gpu")
cpus = jax.devices("cpu")

assert len(cpus) == 6
cpu_id = rank % len(cpus)
cpu = cpus[cpu_id]
jax.config.update("jax_default_device", cpu)

# --- Hyperparameters ---
n = 10
m = 1024     # examples per rank
gamma = 0.05 # learning rate
n_iter = 200 # number of training iterations (epochs)

# --- Ground truth (same on every rank — used only to generate data) ---
key_true = jax.random.PRNGKey(0)
w_true = jax.random.normal(key_true, (n,))

# --- Local training data: distinct per rank ---
key_data = jax.random.PRNGKey(rank + 1)
k1, k2 = jax.random.split(key_data)
x_local = jax.random.normal(k1, (n, m))
y_local = w_true @ x_local + 0.1 * jax.random.normal(k2, (m,)) # corrupted by noise

############ ONLY MODIFY THIS PART OF THE CODE ##############

def loss_fn(w, x, y):
    raise NotImplementedError("You need to implement it!")

# Note the use of a single jax.jit around the whole training step function.
# Advice: temporarily remove jax.jit for debugging
#@jax.jit
def training_step(w, x_local, y_local):
    raise NotImplementedError("You must compute the loss value and the loss gradient here using data-parallel MPI!"
    w_new = w - gamma * grad
    return loss, w_new

##############################################################

# --- Initial weights — must be identical on every rank ---
w = jnp.zeros(n)

# Training loop
for step in range(n_iter):
    loss, w = training_step(w, x_local, y_local)

    if step % 25 == 0 and rank == 0:
        print(f"step {step:4d}  global mean loss = {float(loss):.6f}", flush=True)

if rank == 0:
    err = float(jnp.linalg.norm(w - w_true))
    print(f"\nfinal ||w - w_true|| = {err:.4f}", flush=True)
    print(f"\ntrue:    {w_true}", flush=True)
    print(f"learned: {w}", flush=True)

Overwriting temp.py


In [3]:
#!mpiexec -n 4 python3 temp.py

## Solution

In [4]:
%%writefile temp.py
"""Data-parallel GD on linear regression.

Each rank holds a disjoint subset of training data. After every step,
gradients are averaged across ranks via mpi4jax.allreduce, so every rank
maintains identical weights.
"""

import jax
import jax.numpy as jnp
from mpi4py import MPI
import mpi4jax

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

# Selecting a single CPU and using that by setting the jax_default_device
gpus = jax.devices("gpu")
cpus = jax.devices("cpu")

assert len(cpus) == 6
cpu_id = rank % len(cpus)
cpu = cpus[cpu_id]
jax.config.update("jax_default_device", cpu)

# --- Hyperparameters ---
n = 10
m = 1024     # examples per rank
gamma = 0.05 # learning rate
n_iter = 200 # number of training iterations (epochs)

# --- Ground truth (same on every rank — used only to generate data) ---
key_true = jax.random.PRNGKey(0)
w_true = jax.random.normal(key_true, (n,))

# --- Local training data: distinct per rank ---
key_data = jax.random.PRNGKey(rank + 1)
k1, k2 = jax.random.split(key_data)
x_local = jax.random.normal(k1, (n, m))
y_local = w_true @ x_local + 0.1 * jax.random.normal(k2, (m,)) # corrupted by noise

def loss_fn(w, x, y):
    return jnp.mean((w@x - y) ** 2)

loss_value_and_grad = jax.value_and_grad(loss_fn)

@jax.jit
def training_step(w, x_local, y_local):
    loss, grad = loss_value_and_grad(w, x_local, y_local)
    lossgrad = jnp.concatenate([jnp.array([loss]),grad]) # concatenating to use a single allreduce
    lossgrad_sum = mpi4jax.allreduce(lossgrad, op=MPI.SUM, comm=comm)
    lossgrad_avg = lossgrad_sum/size
    w_new = w - gamma * lossgrad_avg[1:]
    return lossgrad_avg[0], w_new

# --- Initial weights — must be identical on every rank ---
w = jnp.zeros(n)

# Training loop
for step in range(n_iter):
    loss, w = training_step(w, x_local, y_local)

    if step % 25 == 0 and rank == 0:
        print(f"step {step:4d}  global mean loss = {float(loss):.6f}", flush=True)

if rank == 0:
    err = float(jnp.linalg.norm(w - w_true))
    print(f"\nfinal ||w - w_true|| = {err:.4f}", flush=True)
    print(f"\ntrue:    {w_true}", flush=True)
    print(f"learned: {w}", flush=True)

Overwriting temp.py


In [5]:
!mpirun -np 4 python3 temp.py

/opt/tljh/user/envs/hpc4ai/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


step    0  global mean loss = 9.814224
step   25  global mean loss = 0.059539
step   50  global mean loss = 0.010099
step   75  global mean loss = 0.009833
step  100  global mean loss = 0.009832
step  125  global mean loss = 0.009832
step  150  global mean loss = 0.009832
step  175  global mean loss = 0.009832

final ||w - w_true|| = 0.0044

true:    [ 1.6226422   2.0252647  -0.43359444 -0.07861735  0.1760909  -0.97208923
 -0.49529874  0.4943786   0.6643493  -0.9501635 ]
learned: [ 1.6226436   2.024797   -0.43626565 -0.07817512  0.17532372 -0.9721261
 -0.49321917  0.4955785   0.6667251  -0.9504067 ]


## Question 2 - Data-parallel training with equinox and pytrees

In Question 1 the gradient was a single 1D array. For real models — anything beyond a single linear layer — gradients are **PyTrees**: nested structures of arrays (one per parameter). `mpi4jax.allreduce` operates on one tensor at a time, so we first need to flatten the pytree. To compute gradient updates we can use the flattened pytree or `jax.tree.map`.

In this question we will fit the function 

$$y = \sin(2\pi x) + \frac{1}{2}cos(\pi x^2)$$

over the domain $[0, 1]$ with a 2-hidden-layer MLP, using `mpi4jax` for data-parallel training across $4$ ranks.

### a) Handling pytrees with flatten

Modify the script below to complete the implementation. The tricky bit is to handle the pytrees. **Hint:** use the provided `flatten` function.

### b) (Optional, only if you want to learn more about how to use equinox and pytrees)

Modify the script so that `training_step` takes `model` as an input rather than the `mlp_parameters`. You have multiple ways of doing this:

1) Use `@eqx.filter_jit`, compute the gradient descent update by using `ravel_pytree` and then `jax.tree.map`. Then, update the model with `eqx.apply_updates`.

2) Modify `loss_fn` so that it takes `mlp_params` as input (you will need `eqx.partition` and `eqx.combine` and move the definition of `loss_fn` inside the `training_step`), then use `@eqx.filter_jit` and `jax.tree.map` to compute the gradient descent update (if `loss_fn` only takes `mlp_params` as input the gradient will be a pure pytree).

Try them both.

In [6]:
%%writefile temp.py
"""Data-parallel training of an equinox MLP via mpi4jax.allreduce.

Fits y = sin(2*pi*x) + 0.5*cos(pi*x^2) over [0, 1].
Each rank gets a different subset of x; gradients are averaged across
ranks after each step using a single allreduce on the flattened PyTree.
"""

import jax
import jax.numpy as jnp
from jax.flatten_util import ravel_pytree
import equinox as eqx
import mpi4jax
from mpi4py import MPI

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

# Selecting a single CPU and using that by setting the jax_default_device
gpus = jax.devices("gpu")
cpus = jax.devices("cpu")

assert len(cpus) == 6
cpu_id = rank % len(cpus)
cpu = cpus[cpu_id]
jax.config.update("jax_default_device", cpu)

def target_fn(x):
    return jnp.sin(2.0 * jnp.pi * x) + 0.5 * jnp.cos(jnp.pi * x ** 2)

m = 1024
gamma = 5e-3 # learning rate
n_iter = 5000

# --- Per-rank data (distinct samples) ---
key_data = jax.random.PRNGKey(rank + 1)
x_local = jax.random.uniform(key_data, (m, 1))
y_local = target_fn(x_local.squeeze())

# --- Identical model on every rank (same key) ---
model = eqx.nn.MLP(
    in_size=1,
    out_size=1,
    width_size=64,
    depth=2,
    activation=jax.nn.tanh,
    key=jax.random.PRNGKey(0),
)

def flatten(model):
    params, static = eqx.partition(model, eqx.is_array)
    mlp_parameters, unflatten_params = ravel_pytree(params)
    def unflatten(flat_params):
        params = unflatten_params(flat_params)          # pytree of arrays
        return eqx.combine(params, static)

    return mlp_parameters, unflatten

################ ONLY MODIFY THIS PART ##################

mlp_params, unflatten = flatten(model)

@eqx.filter_value_and_grad
def loss_fn(model, x, y):
    pred = jax.vmap(model)(x).squeeze()
    return jnp.mean((pred - y) ** 2)

# For part a) it is sufficient to only modify this function
# Uncomment @jax.jit once you are done with debugging.
#@jax.jit
def training_step(mlp_params, x_local, y_local):
    # Reconstruct the eqx.Module from the flat parameter vector.
    model = unflatten(mlp_params)

    raise NotImplementedError("Need to implement a GD update here!")
    
    return avg_loss, mlp_params_new

# --- Training loop ---
for step in range(n_iter):
    loss, mlp_params = training_step(mlp_params, x_local, y_local)

    if step % 100 == 0 and rank == 0:
        print(f"step {step:4d}  global mean loss = {float(loss):.6f}", flush=True)

model = unflatten(mlp_params)

#############################################################

# --- Final evaluation on rank 0 ---
if rank == 0:
    x_eval = jnp.linspace(0.0, 1.0, 200)
    y_pred = jax.vmap(model)(x_eval[:, None]).squeeze()
    y_true = target_fn(x_eval)
    rmse = float(jnp.sqrt(jnp.mean((y_pred - y_true) ** 2)))
    print(f"\nfinal RMSE on eval grid: {rmse:.6f}")

Overwriting temp.py


## Solution

In [7]:
%%writefile temp.py
"""Data-parallel training of an equinox MLP via mpi4jax.allreduce.

Fits y = sin(2*pi*x) + 0.5*cos(pi*x^2) over [0, 1].
Each rank gets a different subset of x; gradients are averaged across
ranks after each step using a single allreduce on the flattened PyTree.
"""

import jax
import jax.numpy as jnp
from jax.flatten_util import ravel_pytree
import equinox as eqx
import mpi4jax
from mpi4py import MPI

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

# Selecting a single CPU and using that by setting the jax_default_device
gpus = jax.devices("gpu")
cpus = jax.devices("cpu")

assert len(cpus) == 6
cpu_id = rank % len(cpus)
cpu = cpus[cpu_id]
jax.config.update("jax_default_device", cpu)

def target_fn(x):
    return jnp.sin(2.0 * jnp.pi * x) + 0.5 * jnp.cos(jnp.pi * x ** 2)

m = 1024
gamma = 5e-3 # learning rate
n_iter = 5000

# --- Per-rank data (distinct samples) ---
key_data = jax.random.PRNGKey(rank + 1)
x_local = jax.random.uniform(key_data, (m, 1))
y_local = target_fn(x_local.squeeze())

# --- Identical model on every rank (same key) ---
model = eqx.nn.MLP(
    in_size=1,
    out_size=1,
    width_size=64,
    depth=2,
    activation=jax.nn.tanh,
    key=jax.random.PRNGKey(0),
)

def flatten(model):
    params, static = eqx.partition(model, eqx.is_array)
    mlp_parameters, unflatten_params = ravel_pytree(params)
    def unflatten(flat_params):
        params = unflatten_params(flat_params)          # pytree of arrays
        return eqx.combine(params, static)

    return mlp_parameters, unflatten

################ ONLY MODIFY THIS PART ##################

which_part = ["a)", "b1)", "b2)"][0] # modify this to change solutions

if which_part == "a)":
    mlp_params, unflatten = flatten(model)

    @eqx.filter_value_and_grad
    def loss_fn(model, x, y):
        pred = jax.vmap(model)(x).squeeze()
        return jnp.mean((pred - y) ** 2)

    @jax.jit
    def training_step(mlp_params, x_local, y_local):
        # Reconstruct the eqx.Module from the flat parameter vector.
        model = unflatten(mlp_params)

        loss, grads = loss_fn(model, x_local, y_local)

        # Flatten the gradient pytree to match the parameter layout.
        flat_grads, _ = flatten(grads)

        # Single allreduce: loss scalar + flat grads.
        lossgrad = jnp.concatenate([jnp.array([loss]), flat_grads])
        lossgrad_sum = mpi4jax.allreduce(lossgrad, op=MPI.SUM, comm=comm)
        lossgrad_avg = lossgrad_sum / size

        avg_loss = lossgrad_avg[0]
        avg_grads = lossgrad_avg[1:]

        # SGD update directly on the flat vector.
        mlp_params_new = mlp_params - gamma * avg_grads
        return avg_loss, mlp_params_new

    # --- Training loop ---
    for step in range(n_iter):
        loss, mlp_params = training_step(mlp_params, x_local, y_local)

        if step % 100 == 0 and rank == 0:
            print(f"step {step:4d}  global mean loss = {float(loss):.6f}", flush=True)

    model = unflatten(mlp_params)

elif which_part == "b1)":
    @eqx.filter_value_and_grad
    def loss_fn(model, x, y):
        pred = jax.vmap(model)(x).squeeze()
        return jnp.mean((pred - y) ** 2)

    @eqx.filter_jit
    def training_step(model, x_local, y_local):
        loss, grads = loss_fn(model, x_local, y_local)

        # Flatten the gradient pytree into a single vector.
        flat_grads, unflatten = ravel_pytree(grads)

        # Concatenate loss scalar + flat grads → single allreduce.
        lossgrad = jnp.concatenate([jnp.array([loss]), flat_grads])
        lossgrad_sum = mpi4jax.allreduce(lossgrad, op=MPI.SUM, comm=comm)
        lossgrad_avg = lossgrad_sum / size

        # Recover averaged loss and reconstruct the gradient pytree.
        avg_loss = lossgrad_avg[0]
        avg_grads = unflatten(lossgrad_avg[1:])

        # SGD update.
        updates = jax.tree.map(lambda g: -gamma * g, avg_grads)
        model = eqx.apply_updates(model, updates)

        return avg_loss, model

    # --- Training loop ---
    for step in range(n_iter):
        loss, model = training_step(model, x_local, y_local)

        if step % 100 == 0 and rank == 0:
            print(f"step {step:4d}  global mean loss = {float(loss):.6f}", flush=True)
    
elif which_part == "b2)":
    @eqx.filter_jit
    def training_step(model, x_local, y_local):
        params, static = eqx.partition(model, eqx.is_array)

        def loss_from_params(params, x, y):
            m = eqx.combine(params, static)
            pred = jax.vmap(m)(x).squeeze()
            return jnp.mean((pred - y) ** 2)

        loss, grads = jax.value_and_grad(loss_from_params)(params, x_local, y_local)

        flat_grads, unflatten = ravel_pytree(grads)
        lossgrad = jnp.concatenate([jnp.array([loss]), flat_grads])
        lossgrad_sum = mpi4jax.allreduce(lossgrad, op=MPI.SUM, comm=comm)
        lossgrad_avg = lossgrad_sum / size

        avg_loss = lossgrad_avg[0]
        avg_grads = unflatten(lossgrad_avg[1:])

        new_params = jax.tree.map(lambda p, g: p - gamma * g, params, avg_grads)
        model = eqx.combine(new_params, static)
        return avg_loss, model

    # Training loop — model is passed and returned each step
    for step in range(n_iter):
        loss, model = training_step(model, x_local, y_local)

        if step % 100 == 0 and rank == 0:
            print(f"step {step:4d}  global mean loss = {float(loss):.6f}", flush=True)

else: raise ValueError("I do not know which solution you want to use") 
#############################################################

# --- Final evaluation on rank 0 ---
if rank == 0:
    x_eval = jnp.linspace(0.0, 1.0, 200)
    y_pred = jax.vmap(model)(x_eval[:, None]).squeeze()
    y_true = target_fn(x_eval)
    rmse = float(jnp.sqrt(jnp.mean((y_pred - y_true) ** 2)))
    print(f"\nfinal RMSE on eval grid: {rmse:.6f}")

Overwriting temp.py


In [8]:
!mpirun -np 4 python3 temp.py

step    0  global mean loss = 1.075634
step  100  global mean loss = 0.439526
step  200  global mean loss = 0.229343
step  300  global mean loss = 0.213960
step  400  global mean loss = 0.212731
step  500  global mean loss = 0.211928
step  600  global mean loss = 0.211148
step  700  global mean loss = 0.210378
step  800  global mean loss = 0.209611
step  900  global mean loss = 0.208841
step 1000  global mean loss = 0.208062
step 1100  global mean loss = 0.207268
step 1200  global mean loss = 0.206451
step 1300  global mean loss = 0.205604
step 1400  global mean loss = 0.204718
step 1500  global mean loss = 0.203784
step 1600  global mean loss = 0.202792
step 1700  global mean loss = 0.201731
step 1800  global mean loss = 0.200588
step 1900  global mean loss = 0.199348
step 2000  global mean loss = 0.197997
step 2100  global mean loss = 0.196517
step 2200  global mean loss = 0.194888
step 2300  global mean loss = 0.193091
step 2400  global mean loss = 0.191103
step 2500  global mean lo

<hr style="height:4px; background-color:black; border:none; border-color:black;">

# JAX profiling questions

## Question 3 - How to use profiling for optimising code - Radial basis function kernel problem

Consider the radial-basis-function kernel

$$g(x,y)= \exp(-\gamma \lVert x - y \rVert_2^2).$$

Generate $n=64$ random spatial coordinates points in $d=8$ dimensions and store them into a $n$-by-$d$ `jax.numpy` array $X$. Write two JAX routines that compute and return the value of $g(x,y)$ for all point pairs in $X$ by using `for` loops and calling $g$ for each point pair inside the loops (use `jax.jit` on the $g$). By changing the JAX default device run the function on both a CPU and a GPU, then use `jax.profiler.trace` and `xprof` (or `perfetto`) to profile the code. What do you observe? Can you explain why in the GPU case the programme spends a lot of time doing CPU computations? If you wanted to make the function faster what would you do?

Now, make the function faster by using `vmap` twice on an unjitted $g$ and JIT-ing the resulting vmapped function. Run it and profile it again on at least the GPU. Of course it is faster, but do you also observe any difference in the profiler trace compared to the previous naive implementation?

(Optional) Profile it on the CPU as well and see if you can push the vmapped kernel to $n=1024$ and $d=32$.

**Warning:** Stop the computation if it takes too long (more than 3 mins). Double check the zip file size before downloading it. If it is too large processing it on laptops with too little RAM may cause your laptop too start swapping (it's not dangerous, but it freezes). On my linux laptop with 16 GB of RAM chrome is fine up to 100 MB of zip file size. This is not a problem on servers since they have much more memory! P.s. If you use the problem sizes suggested in the question you will be fine!

## Solution

In [9]:
# where we save the trace
jax_trace_folder = "jax_trace"
try: shutil.rmtree(jax_trace_folder) # remove previously saved traces
except FileNotFoundError: pass

def trace_function(fast=False, device=gpus[0]):

    jax.config.update("jax_default_device", device)

    if fast:
        n = 1024
        d = 32
    else:
        n = 64
        d = 8
        
    gamma = 1.0 / d

    key = jax.random.PRNGKey(0)
    x = jax.random.normal(key, (n, d), dtype=jnp.float32)

    def rbf(x, y):
        return jnp.exp(-gamma * jnp.sum((x - y) ** 2))

    rbf_jit = jax.jit(rbf)

    if fast:
        @jax.jit
        def kernel(x):
            return jax.vmap(lambda xi: jax.vmap(lambda xj: rbf(xi, xj))(x))(x)
    else:
        def kernel(x):
            rows = []
            for i in range(x.shape[0]):
                row = []
                for j in range(x.shape[0]):
                    row.append(rbf_jit(x[i], x[j]))
                rows.append(jnp.stack(row))
            return jnp.stack(rows)

    print("Running " + ["slow","fast"][fast] + " kernel")
    y = kernel(x)
    jax.block_until_ready(y)

    print("Starting trace...")
    with jax.profiler.trace(jax_trace_folder, create_perfetto_trace=True, create_perfetto_link=False):
        y = kernel(x)
        jax.block_until_ready(y)

    # download, unzip, and open with xprof -l ./jax_trace or with https://ui.perfetto.dev/
    shutil.make_archive(jax_trace_folder, "zip", root_dir=".", base_dir=jax_trace_folder) 
    print("Profile trace saved!")

if __name__ == "__main__":
    trace_function(fast=True, device=gpus[0])

    # In the GPU slow-kernel case the programme spends a lot of time doing CPU computations since the only function that is actually called on the GPU is rbf.
    # Everything else happens on the CPU and the CPU has to handle all the host/device memory movement for the inputs/outputs of the rbf function.
    # If instead we JIT the vmap kernel the whole kernel is run on the GPU so the CPU does much less work, there is much less memory movement,
    # and everything is much faster.

Running fast kernel
Starting trace...
Profile trace saved!


## (Optional) Question 4 - Profiling with sharding

Try rerunning the example in the previous problem by using the fast kernel and sharding first across 4 CPUs and then across 2 GPUS (using whatever parallelism strategy you prefer). Profile the results and analyse them with `xprof` (or `perfetto`). Is it faster than without sharding? Try playing with different mesh shapes.